# เทรนโมเดล CatBoost บนชุด training ราย 6 ชั่วโมง (RES002) — v2 (จัดเรียงใหม่)

รวมผลจากรอบทดสอบก่อนหน้าเข้ามาเป็น feature set สุดท้าย (ตัดเซลทดลอง ablation/addition ออกแล้ว เพราะ
ตัดสินใจแล้ว) ก่อนรันจริง:

1. **ตัด `is_release_active` + `release_rate_daily (m3/day)` ออก** — ablation test ยืนยันว่าตัดออกไม่กระทบ
   walk-forward CV NSE เลย (mean delta = -0.0014) เพราะโมเดลเห็นผลของ release อยู่แล้วผ่าน
   Water_Level_t/Storage_S_t/DeltaS_t — **เก็บ `API_t` ไว้** ตามที่ขอ (ยังไม่ได้ทดสอบตัด)
2. **เพิ่ม `Rain_lag1`/`Rain_lag2` เข้า training ถาวรแล้ว** (อยู่ใน `Training_6hourly_full.csv` แล้ว)
   — addition test รอบก่อนพบว่าช่วยชัดเจนที่ horizon สั้น (H1-H4) แต่ทำให้แย่ลงที่ horizon ยาว (H5-H12)
   ตอนนั้น**ยังไม่ได้ re-tune** ให้เหมาะกับ feature ใหม่ — รอบนี้จะ **re-tune ทุก horizon ใหม่ทั้งหมด**
   (Optuna จะเลือกเองว่า horizon ไหนควรพึ่ง Rain_lag มากน้อยแค่ไหนผ่าน regularization/depth ที่ tune ใหม่)
3. **`OUTDIR` เปลี่ยนเป็น `outputs_6hourly_catboost_v2`** (โฟลเดอร์ใหม่ ไม่ใช้ checkpoint เดิมของรอบก่อน) —
   **สำคัญ**: เพราะ feature set เปลี่ยนไปจากรอบก่อน (ตัด 2 เพิ่ม 2) ถ้าใช้ checkpoint เดิมต่อ ระบบ
   resume จะข้าม horizon ที่ "เคย" tune ไปแล้วโดยไม่รู้ว่า feature set เปลี่ยน ทำให้ได้ params ที่ tune จาก
   feature set ผิดชุดมาปนกัน จึงต้องขึ้นโฟลเดอร์ใหม่เพื่อความถูกต้อง

**สถาปัตยกรรม**: เหมือนเดิมทุกประการ — direct delta-regression ต่อ horizon (CatBoost ทำนาย delta =
Q_in_t+h - Q_in_t แล้ว `prediction = clip(Q_in_t + delta_pred, 0, None)`), 12 horizon (y1-y12 = 6h-72h
ข้างหน้า), chronological wet/dry/wet 70/15/15 split, walk-forward CV (TimeSeriesSplit 5-fold) เป็นตัวชี้ขาด
หลัก, hardware benchmark CPU vs GPU ก่อนเทรนจริง

**Checkpoint/resume**: ผลลัพธ์เขียนลง `outputs_6hourly_catboost_v2/*.csv` ทีละ horizon — ปิด kernel/รันซ้ำ
กลางทางได้ รันใหม่จะข้ามส่วนที่เสร็จแล้วเอง

**สารบัญ**: Cell 1 ติดตั้ง lib | Cell 2 setup | Cell 3-4 hardware benchmark CPU vs GPU | Cell 5-6 โหลด+split
ข้อมูล | Cell 7 Optuna tuning (checkpointed) | Cell 8 walk-forward CV (checkpointed) | Cell 9 สรุปผล | Cell 10
export .pkl + metadata | Cell 11 feature importance

## Cell 1 — ติดตั้ง library (รันครั้งเดียว ข้ามได้ถ้าติดตั้งไว้แล้ว)

ถ้าใช้ venv/conda แยกอยู่แล้ว รันใน terminal แทนก็ได้: `pip install catboost optuna pandas numpy scikit-learn`

**GPU**: ถ้าต้องการลองเทรนบน GPU ต้องมี NVIDIA GPU + CUDA driver ติดตั้งอยู่แล้วในเครื่อง (CatBoost ไม่รองรับ
GPU ของ AMD/Intel) — ไม่ต้องติดตั้งอะไรเพิ่มสำหรับ CatBoost เอง ถ้าไม่มี GPU ที่ใช้ได้ notebook นี้จะ fallback
ไปใช้ CPU อัตโนมัติ (ดู Cell 3-4)

In [1]:
import sys
!{sys.executable} -m pip install catboost optuna pandas numpy scikit-learn --quiet

## Cell 2 — Setup: imports, CatBoost adapter (parameterize by device), metric functions

In [2]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTDIR = "outputs_6hourly_catboost_v2"  # โฟลเดอร์ใหม่ -- feature set เปลี่ยนจากรอบก่อน (ดู Cell แรก)
os.makedirs(OUTDIR, exist_ok=True)
PERF_CSV = os.path.join(OUTDIR, "tuned_test_performance.csv")
PARAMS_CSV = os.path.join(OUTDIR, "tuned_params.csv")
CV_CSV = os.path.join(OUTDIR, "cv_stability_diagnostic.csv")
CSV_PATH = "Training_6hourly_full.csv"  # ต้องอยู่โฟลเดอร์เดียวกับ notebook นี้ (มี Rain_lag1/2 แล้ว)

# feature ที่ตัดสินใจตัดออกแล้ว (ablation test ยืนยันไม่กระทบผล) -- ยังเก็บไว้ในไฟล์ CSV เผื่ออ้างอิง/ตรวจสอบ
# แต่ไม่ใช้เป็น input ให้โมเดล
EXCLUDE_FEATURES = {"is_release_active", "release_rate_daily (m3/day)"}


class CatBoostAdapter:
    """เหมือน adapter เดิมของโปรเจกต์ทุกประการ แค่เพิ่ม task_type/devices ให้สลับ CPU<->GPU ได้
    ผ่านตัวแปร global CATBOOST_TASK_TYPE (ตั้งค่าใน Cell 3-4 หลัง benchmark)"""
    name = "CatBoost"

    @staticmethod
    def build(params, with_early_stopping=True, task_type=None):
        p = dict(params)
        p.setdefault("loss_function", "MAE")
        p.setdefault("random_seed", 42)
        p.setdefault("verbose", False)
        tt = task_type if task_type is not None else globals().get("CATBOOST_TASK_TYPE", "CPU")
        p["task_type"] = tt
        if tt == "GPU":
            p["devices"] = "0"
            p.setdefault("bootstrap_type", "Bayesian")
        else:
            p.setdefault("bootstrap_type", "Bernoulli")
            p.setdefault("subsample", p.get("subsample", 0.8))
        if with_early_stopping:
            p.setdefault("od_type", "Iter")
            p.setdefault("od_wait", 200)
        return CatBoostRegressor(**p)

    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
            return model, model.get_best_iteration()
        model.fit(X_tr, y_tr, verbose=False)
        return model, model.tree_count_

    @staticmethod
    def predict(model, X):
        return model.predict(X)


def mae_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))


def rmse_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))


def nse_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(1 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-9))


def kge_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    r = np.corrcoef(y, yhat)[0, 1]
    alpha = (yhat.std() + 1e-9) / (y.std() + 1e-9)
    beta = (yhat.mean() + 1e-9) / (y.mean() + 1e-9)
    return float(1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))


print("Setup OK")

Setup OK


## Cell 3-4 — Hardware benchmark: CPU vs GPU จริงบนเครื่องนี้

เทรนโมเดลตัวอย่างขนาดจริงบน device ทั้งสอง วัดเวลาจริง แล้วเลือกตัวที่เร็วกว่าอัตโนมัติให้ cell ถัดไปทั้งหมด
ใช้ (ข้อมูลชุดนี้เล็ก ~1,500 แถว มักจะ CPU เร็วกว่าเพราะ overhead ของ GPU แต่ให้วัดจริงแทนเดา)

ตั้ง `FORCE_DEVICE = "CPU"` หรือ `"GPU"` ถ้าอยากบังคับเอง ข้าม benchmark ได้

In [3]:
FORCE_DEVICE = None  # ตั้งเป็น "CPU" หรือ "GPU" ถ้าอยากบังคับเอง ข้าม benchmark ด้านล่าง

_bench_df = pd.read_csv(CSV_PATH)
_bench_qin_col = "Q_in_t (m3/6h)"
_bench_targets = sorted([c for c in _bench_df.columns if re.match(r"^y\d+=Q_in", str(c))],
                         key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
_bench_exclude = {"Datetime"} | set(_bench_targets) | EXCLUDE_FEATURES
_bench_feat_cols = [c for c in _bench_df.columns if c not in _bench_exclude]
_bench_mask = ~_bench_df[_bench_targets + _bench_feat_cols].isna().any(axis=1)
_bench_X = _bench_df.loc[_bench_mask, _bench_feat_cols].reset_index(drop=True)
_bench_y = (_bench_df.loc[_bench_mask, _bench_targets[0]].to_numpy()
            - _bench_df.loc[_bench_mask, _bench_qin_col].to_numpy())
print(f"Benchmark data: {len(_bench_X)} rows x {len(_bench_feat_cols)} features")

_bench_params = dict(iterations=650, learning_rate=0.011, depth=6, l2_leaf_reg=20,
                      min_data_in_leaf=8, random_strength=1.0)

results = {}
if FORCE_DEVICE is not None:
    print(f"FORCE_DEVICE={FORCE_DEVICE} -- ข้าม benchmark")
    CATBOOST_TASK_TYPE = FORCE_DEVICE
else:
    t0 = time.time()
    m_cpu = CatBoostAdapter.build(_bench_params, with_early_stopping=False, task_type="CPU")
    m_cpu, _ = CatBoostAdapter.fit(m_cpu, _bench_X, _bench_y)
    cpu_time = time.time() - t0
    results["CPU"] = cpu_time
    print(f"CPU:  {cpu_time:.2f} s")

    try:
        t0 = time.time()
        m_gpu = CatBoostAdapter.build(_bench_params, with_early_stopping=False, task_type="GPU")
        m_gpu, _ = CatBoostAdapter.fit(m_gpu, _bench_X, _bench_y)
        gpu_time = time.time() - t0
        results["GPU"] = gpu_time
        print(f"GPU:  {gpu_time:.2f} s")
    except Exception as e:
        print(f"GPU: ใช้ไม่ได้บนเครื่องนี้ ({type(e).__name__}: {str(e)[:200]})")
        print("-> ไม่มี CUDA GPU ที่ CatBoost มองเห็น หรือ driver ไม่รองรับ -- ใช้ CPU ต่อไป")

    if "GPU" in results and results["GPU"] < results["CPU"]:
        CATBOOST_TASK_TYPE = "GPU"
        speedup = results["CPU"] / results["GPU"]
        print(f"\n=> เลือก GPU (เร็วกว่า CPU {speedup:.2f}x บนข้อมูลชุดนี้)")
    else:
        CATBOOST_TASK_TYPE = "CPU"
        if "GPU" in results:
            slowdown = results["GPU"] / results["CPU"]
            print(f"\n=> เลือก CPU (GPU ช้ากว่า CPU {slowdown:.2f}x บนข้อมูลชุดนี้)")
        else:
            print("\n=> เลือก CPU (ไม่มี GPU ที่ใช้ได้)")

print(f"\nCATBOOST_TASK_TYPE = {CATBOOST_TASK_TYPE!r} -- cell ถัดไปทั้งหมดจะใช้ค่านี้")

Benchmark data: 1483 rows x 15 features
CPU:  0.90 s


Default metric period is 5 because MAE is/are not implemented for GPU


GPU:  5.06 s

=> เลือก CPU (GPU ช้ากว่า CPU 5.62x บนข้อมูลชุดนี้)

CATBOOST_TASK_TYPE = 'CPU' -- cell ถัดไปทั้งหมดจะใช้ค่านี้


## Cell 5-6 — โหลดข้อมูล + chronological wet/dry/wet split (70/15/15)

`feature_cols` คำนวณอัตโนมัติจากทุกคอลัมน์ใน CSV ยกเว้น `Datetime`, target (y1-y12), และ
`EXCLUDE_FEATURES` (is_release_active, release_rate_daily ที่ตัดสินใจตัดออกแล้ว) — `Rain_lag1`/`Rain_lag2`
รวมอยู่ในนี้อัตโนมัติเพราะอยู่ใน CSV แล้วและไม่ได้อยู่ใน exclude list

In [4]:
df = pd.read_csv(CSV_PATH)
df["Datetime_dt"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.sort_values("Datetime_dt").reset_index(drop=True)
print("Datetime range:", df["Datetime_dt"].min(), "to", df["Datetime_dt"].max(), "| Rows:", len(df))

QIN_COL = "Q_in_t (m3/6h)"
targets = sorted([c for c in df.columns if re.match(r"^y\d+=Q_in", str(c))],
                  key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
print("Targets:", targets)

exclude = {"Datetime", "Datetime_dt"} | set(targets) | EXCLUDE_FEATURES
feature_cols = [c for c in df.columns if c not in exclude]
print(f"Features ({len(feature_cols)}):", feature_cols)

X = df[feature_cols].copy()
Y = df[targets].copy()
mask = (~Y.isna().any(axis=1)) & (~X.isna().any(axis=1)) & (~df["Datetime_dt"].isna())
X, Y = X.loc[mask].reset_index(drop=True), Y.loc[mask].reset_index(drop=True)
dates = df.loc[mask, "Datetime_dt"].reset_index(drop=True)
N = len(dates)
print("Rows used (after dropping NaN lag/target rows):", N)

dry_start, dry_end = pd.Timestamp("2026-02-03"), pd.Timestamp("2026-05-17")
wet_before_idx = np.where(dates < dry_start)[0]
dry_idx = np.where((dates >= dry_start) & (dates <= dry_end))[0]
wet_after_idx = np.where(dates > dry_end)[0]
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15


def chrono_split(idx_array, frac_train=TRAIN_FRAC, frac_val=VAL_FRAC):
    n = len(idx_array)
    c1 = int(round(n * frac_train))
    c2 = int(round(n * (frac_train + frac_val)))
    return idx_array[:c1], idx_array[c1:c2], idx_array[c2:]


wb_train, wb_val, wb_test = chrono_split(wet_before_idx)
dry_train, dry_val, dry_test = chrono_split(dry_idx)
wa_train, wa_val, wa_test = chrono_split(wet_after_idx)

train_idx = np.sort(np.concatenate([wb_train, dry_train, wa_train]))
val_idx = np.sort(np.concatenate([wb_val, dry_val, wa_val]))
test_idx = np.sort(np.concatenate([wb_test, dry_test, wa_test]))

X_train, Y_train_abs = X.iloc[train_idx].reset_index(drop=True), Y.iloc[train_idx].reset_index(drop=True)
X_val, Y_val_abs = X.iloc[val_idx].reset_index(drop=True), Y.iloc[val_idx].reset_index(drop=True)
X_test, Y_test_abs = X.iloc[test_idx].reset_index(drop=True), Y.iloc[test_idx].reset_index(drop=True)
dates_test = dates.iloc[test_idx].reset_index(drop=True)
base_test = X_test[QIN_COL].to_numpy()
print(f"Split sizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

Datetime range: 2025-06-27 07:00:00 to 2026-07-31 07:00:00 | Rows: 1537
Targets: ['y1=Q_in_t+1 (m3/6h)', 'y2=Q_in_t+2 (m3/6h)', 'y3=Q_in_t+3 (m3/6h)', 'y4=Q_in_t+4 (m3/6h)', 'y5=Q_in_t+5 (m3/6h)', 'y6=Q_in_t+6 (m3/6h)', 'y7=Q_in_t+7 (m3/6h)', 'y8=Q_in_t+8 (m3/6h)', 'y9=Q_in_t+9 (m3/6h)', 'y10=Q_in_t+10 (m3/6h)', 'y11=Q_in_t+11 (m3/6h)', 'y12=Q_in_t+12 (m3/6h)']
Features (15): ['Q_in_t (m3/6h)', 'Water_Level_t (m)', 'Storage_S_t (m3)', 'DeltaS_t (m3/6h)', '%Full_t', 'Rain_obs_t (mm)', 'API_t (mm)', 'spill_m3', 'Qin_lag1 (m3/6h)', 'Qin_lag2 (m3/6h)', 'Rain_lag1 (mm)', 'Rain_lag2 (mm)', 'Rain_roll3 (mm)', 'Rain_roll5 (mm)', 'Rain_roll7 (mm)']
Rows used (after dropping NaN lag/target rows): 1483
Split sizes: train=1038, val=222, test=223


## Cell 7 — Optuna tuning ต่อ horizon (checkpointed, ใช้ device จาก benchmark)

re-tune ใหม่ทั้งหมดด้วย feature set ใหม่ (`N_TRIALS` ปรับได้ตามเวลาที่มี)

In [5]:
N_TRIALS = 120
INNER_SPLITS = 4


def make_optuna_params(trial):
    return dict(
        iterations=trial.suggest_int("iterations", 300, 1500, step=20),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        depth=trial.suggest_int("depth", 2, 6),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 500.0, log=True),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 3, 40),
        random_strength=trial.suggest_float("random_strength", 0.0, 5.0),
    )


def make_objective(X_tr, y_tr_delta, qin_tr):
    tscv_inner = TimeSeriesSplit(n_splits=INNER_SPLITS)

    def objective(trial):
        params = make_optuna_params(trial)
        fold_rmses = []
        for tr_idx, te_idx in tscv_inner.split(X_tr):
            Xtr_f, Xte_f = X_tr.iloc[tr_idx], X_tr.iloc[te_idx]
            ytr_f, yte_f = y_tr_delta[tr_idx], y_tr_delta[te_idx]
            qin_te_f = qin_tr[te_idx]
            m = CatBoostAdapter.build(params, with_early_stopping=False)
            m, _ = CatBoostAdapter.fit(m, Xtr_f, ytr_f)
            y_pred_abs_f = np.clip(qin_te_f + CatBoostAdapter.predict(m, Xte_f), 0, None)
            y_true_abs_f = yte_f + qin_te_f
            fold_rmses.append(rmse_np(y_true_abs_f, y_pred_abs_f))
        return float(np.mean(fold_rmses))

    return objective


done_h = set()
if os.path.exists(PERF_CSV):
    prev = pd.read_csv(PERF_CSV)
    done_h = set(prev["H"])
    print(f"Resuming: {len(done_h)} horizon ทำเสร็จแล้ว")

t0 = time.time()
qin_train_arr = X_train[QIN_COL].to_numpy()

for h, tcol in enumerate(targets, start=1):
    if h in done_h:
        continue
    y_train_delta = Y_train_abs[tcol].to_numpy() - qin_train_arr

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(X_train, y_train_delta, qin_train_arr), n_trials=N_TRIALS)
    best_params = study.best_params

    yval_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
    m = CatBoostAdapter.build(best_params, with_early_stopping=True)
    m, best_iter = CatBoostAdapter.fit(m, X_train, y_train_delta, X_val, yval_delta)

    delta_pred_test = CatBoostAdapter.predict(m, X_test)
    q_pred_test = np.clip(base_test + delta_pred_test, 0, None)
    y_true_test = Y_test_abs[tcol].to_numpy()
    delta_pred_val = CatBoostAdapter.predict(m, X_val)
    q_pred_val = np.clip(X_val[QIN_COL].to_numpy() + delta_pred_val, 0, None)

    perf_row = {
        "H": h, "Target": tcol, "best_iter": best_iter, "inner_cv_rmse": study.best_value,
        "Baseline_MAE": mae_np(y_true_test, base_test), "Test_MAE": mae_np(y_true_test, q_pred_test),
        "Baseline_RMSE": rmse_np(y_true_test, base_test), "Test_RMSE": rmse_np(y_true_test, q_pred_test),
        "Baseline_NSE": nse_np(y_true_test, base_test), "Test_NSE": nse_np(y_true_test, q_pred_test),
        "Test_KGE": kge_np(y_true_test, q_pred_test),
        "Val_NSE": nse_np(Y_val_abs[tcol].to_numpy(), q_pred_val),
    }
    params_row = {"H": h, "Target": tcol, "inner_cv_rmse": study.best_value, **best_params}
    pd.DataFrame([perf_row]).to_csv(PERF_CSV, mode="a", header=not os.path.exists(PERF_CSV), index=False)
    pd.DataFrame([params_row]).to_csv(PARAMS_CSV, mode="a", header=not os.path.exists(PARAMS_CSV), index=False)
    print(f"[{time.time()-t0:.0f}s] H{h}: Test_NSE={perf_row['Test_NSE']:.4f} "
          f"(baseline={perf_row['Baseline_NSE']:.4f}), Val_NSE={perf_row['Val_NSE']:.4f}")

print(f"\nTuning done, {time.time()-t0:.0f}s total")

[267s] H1: Test_NSE=-1.2390 (baseline=-0.6302), Val_NSE=-0.0664
[505s] H2: Test_NSE=-0.1907 (baseline=-0.5457), Val_NSE=0.0679
[745s] H3: Test_NSE=-0.0367 (baseline=-0.8506), Val_NSE=0.1478
[987s] H4: Test_NSE=-0.0715 (baseline=-0.1077), Val_NSE=0.0941
[1123s] H5: Test_NSE=-0.2399 (baseline=-0.8609), Val_NSE=-0.1340
[1271s] H6: Test_NSE=-0.2907 (baseline=-0.6488), Val_NSE=-0.0008
[1417s] H7: Test_NSE=-0.3045 (baseline=-0.7862), Val_NSE=0.1002
[1637s] H8: Test_NSE=-0.0350 (baseline=-0.4366), Val_NSE=0.2106
[1786s] H9: Test_NSE=-0.3022 (baseline=-0.8301), Val_NSE=-0.1971
[2014s] H10: Test_NSE=-0.1016 (baseline=-0.6289), Val_NSE=-0.2387
[2156s] H11: Test_NSE=-0.1004 (baseline=-0.8493), Val_NSE=-0.0186
[2281s] H12: Test_NSE=-0.3039 (baseline=-0.4571), Val_NSE=0.0732

Tuning done, 2281s total


## Cell 8 — walk-forward CV diagnostic (checkpointed)

สัญญาณหลักที่ควรอ้างอิง (ไม่ใช่ Test_NSE จาก single split) — ต้องรัน Cell 7 ให้ครบ 12/12 horizon ก่อน

In [6]:
tuned_params_df = pd.read_csv(PARAMS_CSV)
int_cols = ["iterations", "depth", "min_data_in_leaf"]

done_h_cv = set()
if os.path.exists(CV_CSV):
    prev = pd.read_csv(CV_CSV)
    done_h_cv = set(prev["H"])
    print(f"Resuming: {len(done_h_cv)} horizon ทำเสร็จแล้ว")

t1 = time.time()
tscv = TimeSeriesSplit(n_splits=5)

for h, tcol in enumerate(targets, start=1):
    if h in done_h_cv:
        continue
    tuned_row = tuned_params_df[tuned_params_df["H"] == h].iloc[0]
    param_cols = [c for c in tuned_params_df.columns if c not in ("H", "Target", "inner_cv_rmse")]
    params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
    for k in int_cols:
        if k in params:
            params[k] = int(params[k])

    fold_rows = []
    for fold, (tr, te) in enumerate(tscv.split(X_train), start=1):
        Xtr_cv, Xte_cv = X_train.iloc[tr].reset_index(drop=True), X_train.iloc[te].reset_index(drop=True)
        base_te_cv = Xte_cv[QIN_COL].to_numpy()
        ytr_abs_cv = Y_train_abs[tcol].iloc[tr].reset_index(drop=True).to_numpy()
        yte_abs_cv = Y_train_abs[tcol].iloc[te].reset_index(drop=True).to_numpy()
        ytr_delta_cv = ytr_abs_cv - Xtr_cv[QIN_COL].to_numpy()
        m_cv = CatBoostAdapter.build(params, with_early_stopping=False)
        m_cv, _ = CatBoostAdapter.fit(m_cv, Xtr_cv, ytr_delta_cv)
        yhat_cv = np.clip(base_te_cv + CatBoostAdapter.predict(m_cv, Xte_cv), 0, None)
        fold_rows.append({"fold": fold, "H": h,
                           "Baseline_NSE": nse_np(yte_abs_cv, base_te_cv),
                           "Model_NSE": nse_np(yte_abs_cv, yhat_cv)})

    pd.DataFrame(fold_rows).to_csv(CV_CSV, mode="a", header=not os.path.exists(CV_CSV), index=False)
    mean_nse = np.mean([r["Model_NSE"] for r in fold_rows])
    print(f"[{time.time()-t1:.0f}s] H{h}: walkforward_cv_NSE mean={mean_nse:.4f}")

print(f"\nWalk-forward CV done, {time.time()-t1:.0f}s total")

[3s] H1: walkforward_cv_NSE mean=0.3600
[5s] H2: walkforward_cv_NSE mean=0.1464
[7s] H3: walkforward_cv_NSE mean=-0.1004
[9s] H4: walkforward_cv_NSE mean=0.0122
[10s] H5: walkforward_cv_NSE mean=-0.0207
[11s] H6: walkforward_cv_NSE mean=-0.0875
[12s] H7: walkforward_cv_NSE mean=-0.1392
[13s] H8: walkforward_cv_NSE mean=-0.3602
[15s] H9: walkforward_cv_NSE mean=-0.1849
[17s] H10: walkforward_cv_NSE mean=-0.2126
[18s] H11: walkforward_cv_NSE mean=-0.2759
[19s] H12: walkforward_cv_NSE mean=-0.3444

Walk-forward CV done, 19s total


## Cell 9 — สรุปผล walk-forward CV NSE ต่อ horizon (v2) เทียบกับ v1 (รอบก่อน) และโมเดลรายวัน production

เทียบ 3 อย่าง: v2 (feature set ใหม่, re-tuned), v1 (feature set เดิม, ผลจากรอบก่อน — กรอกด้วยมือด้านล่าง),
และโมเดลรายวัน production ที่จุดเทียบได้ (24h/48h/72h)

In [7]:
cv_df = pd.read_csv(CV_CSV)
perf_df = pd.read_csv(PERF_CSV)
cv_summary = cv_df.groupby("H")["Model_NSE"].mean().reset_index()
cv_summary["Baseline_NSE_test"] = perf_df.set_index("H").loc[cv_summary["H"], "Baseline_NSE"].to_numpy()
cv_summary["hours_ahead"] = cv_summary["H"] * 6

print("=== walk-forward CV NSE (mean of 5 folds) ต่อ horizon -- v2 (ตัด release feat, เพิ่ม Rain_lag, re-tuned) ===")
print(cv_summary[["H", "hours_ahead", "Model_NSE", "Baseline_NSE_test"]].round(4).to_string(index=False))

# ผลรอบก่อน (v1, ไม่มี Rain_lag, มี release feature, ไม่ retuned สำหรับ Rain_lag) -- อ้างอิงเทียบเฉยๆ
V1_WF_CV_NSE = {1: 0.2933, 2: 0.0490, 3: 0.0063, 4: -0.1956, 5: -0.0471, 6: -0.1095,
                7: -0.1126, 8: -0.1265, 9: -0.1433, 10: -0.1547, 11: -0.1678, 12: -0.2968}
DAILY_PROD_WF_CV_NSE_AT_24_48_72H = {4: 0.4216, 8: 0.2559, 12: -0.0649}  # H4=24h, H8=48h, H12=72h เทียบกับ daily H1/H2/H3

print("\n=== เทียบ v2 vs v1 (รอบก่อน) ===")
for _, row in cv_summary.iterrows():
    h = int(row["H"])
    v1 = V1_WF_CV_NSE.get(h)
    v2 = row["Model_NSE"]
    delta = v2 - v1 if v1 is not None else None
    daily_note = ""
    if h in DAILY_PROD_WF_CV_NSE_AT_24_48_72H:
        daily_note = f" | โมเดลรายวัน (จุดเดียวกัน) = {DAILY_PROD_WF_CV_NSE_AT_24_48_72H[h]:+.4f}"
    print(f"H{h:>2} ({h*6:>2}h): v1={v1:+.4f} -> v2={v2:+.4f} (delta={delta:+.4f}){daily_note}")

=== walk-forward CV NSE (mean of 5 folds) ต่อ horizon -- v2 (ตัด release feat, เพิ่ม Rain_lag, re-tuned) ===
 H  hours_ahead  Model_NSE  Baseline_NSE_test
 1            6     0.3600            -0.6302
 2           12     0.1464            -0.5457
 3           18    -0.1004            -0.8506
 4           24     0.0122            -0.1077
 5           30    -0.0207            -0.8609
 6           36    -0.0875            -0.6488
 7           42    -0.1392            -0.7862
 8           48    -0.3602            -0.4366
 9           54    -0.1849            -0.8301
10           60    -0.2126            -0.6289
11           66    -0.2759            -0.8493
12           72    -0.3444            -0.4571

=== เทียบ v2 vs v1 (รอบก่อน) ===
H 1 ( 6h): v1=+0.2933 -> v2=+0.3600 (delta=+0.0667)
H 2 (12h): v1=+0.0490 -> v2=+0.1464 (delta=+0.0974)
H 3 (18h): v1=+0.0063 -> v2=-0.1004 (delta=-0.1067)
H 4 (24h): v1=-0.1956 -> v2=+0.0122 (delta=+0.2078) | โมเดลรายวัน (จุดเดียวกัน) = +0.4216
H 5 (30h): v1

## Cell 10 — Export โมเดลเป็น .pkl + metadata (checkpoint ผลรอบนี้)

รันต่อจาก Cell 2-8 ในเคอร์เนลเดิมได้เลย **ถ้า restart kernel มาใหม่ ต้องรัน Cell 2, 5-6 ก่อน Cell นี้**
(ไม่ต้องรัน Cell 3-4/7/8 ใหม่ เพราะอ่าน tuned params จาก `outputs_6hourly_catboost_v2/*.csv` ที่บันทึกไว้
แล้ว) — **ไม่ได้ wire เข้า production ใดๆ** เป็นแค่ checkpoint ในโฟลเดอร์ experiments นี้ ตามที่ตกลงกันไว้ว่า
จะไม่แตะโมเดลที่ deploy อยู่จนกว่าจะบอกว่า final

In [ ]:
import pickle
import json
from datetime import datetime

EXPORT_DIR = "exported_v2_" + datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(EXPORT_DIR, exist_ok=True)

assert "X_train" in dir(), "X_train ไม่พบในหน่วยความจำ -- รัน Cell 2 แล้ว Cell 5-6 (โหลด+split ข้อมูล) ก่อน"

tuned_params_df = pd.read_csv(PARAMS_CSV)
cv_df = pd.read_csv(CV_CSV)
perf_df = pd.read_csv(PERF_CSV)
cv_summary = cv_df.groupby("H")["Model_NSE"].mean().reset_index().set_index("H")["Model_NSE"]

int_cols = ["iterations", "depth", "min_data_in_leaf"]

fitted_models = {}
choice_meta = {}
for h, tcol in enumerate(targets, start=1):
    tuned_row = tuned_params_df[tuned_params_df["H"] == h].iloc[0]
    param_cols = [c for c in tuned_params_df.columns if c not in ("H", "Target", "inner_cv_rmse")]
    params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
    for k in int_cols:
        if k in params:
            params[k] = int(params[k])

    y_train_delta = Y_train_abs[tcol].to_numpy() - X_train[QIN_COL].to_numpy()
    y_val_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
    m = CatBoostAdapter.build(params, with_early_stopping=True)
    m, best_iter = CatBoostAdapter.fit(m, X_train, y_train_delta, X_val, y_val_delta)
    fitted_models[h] = m

    perf_row = perf_df[perf_df["H"] == h].iloc[0]
    choice_meta[str(h)] = {
        "model_name": "CatBoost",
        "target_column": tcol,
        "hours_ahead": h * 6,
        "walkforward_cv_nse": float(cv_summary.loc[h]),
        "test_nse": float(perf_row["Test_NSE"]),
        "val_nse": float(perf_row["Val_NSE"]),
        "best_iter": int(best_iter),
        "tuned_params": {k: (int(v) if k in int_cols else float(v)) for k, v in params.items()},
    }
    print(f"H{h} ({h*6}h ahead): walkforward_cv_nse={cv_summary.loc[h]:.4f}, test_nse={perf_row['Test_NSE']:.4f}")

pkl_path = os.path.join(EXPORT_DIR, "deployment_regressors_6hourly_v2.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(fitted_models, f)

meta = {
    "exported_at": datetime.now().isoformat(),
    "architecture": "direct delta-regression per horizon, CatBoost only, 6-hourly resolution, v2 "
                     "(dropped is_release_active/release_rate_daily, added Rain_lag1/Rain_lag2, re-tuned) "
                     "(EXPERIMENTAL -- ไม่ใช่ production ปัจจุบันซึ่งเป็นรายวัน)",
    "training_data_source": "Training_6hourly_full.csv",
    "catboost_task_type": CATBOOST_TASK_TYPE,
    "excluded_features": sorted(EXCLUDE_FEATURES),
    "feature_cols": list(X_train.columns),
    "qin_col": QIN_COL,
    "targets": targets,
    "split_method": "chronological 70/15/15 within 3 season segments (wet_before/dry/wet_after), "
                     "dry window 2026-02-03 to 2026-05-17",
    "prediction_note": "predict delta from current Q_in_t (m3/6h), then q_pred = clip(Q_in_t + delta_pred, 0, None)",
    "deployment_model_per_horizon": choice_meta,
}
meta_path = os.path.join(EXPORT_DIR, "model_metadata_6hourly_v2.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\nExported to: {EXPORT_DIR}/")
print(f"  {pkl_path}")
print(f"  {meta_path}")
print("\nยังไม่ได้ wire เข้า production -- เป็นแค่ checkpoint ของผลรอบนี้ในโฟลเดอร์ experiments")

## Cell 11 — Feature importance ต่อ horizon (v2)

เหมือนเดิม ใช้ CatBoost `get_feature_importance()` (PredictionValuesChange) ต่อ horizon แล้วเฉลี่ยรวม 12
horizon — ดูว่า Rain_lag1/2 ที่เพิ่งเพิ่มเข้ามาจริงๆ มี importance เท่าไหร่หลัง re-tune

In [ ]:
assert "X_train" in dir(), "X_train ไม่พบในหน่วยความจำ -- รัน Cell 2 แล้ว Cell 5-6 ก่อน"

tuned_params_df = pd.read_csv(PARAMS_CSV)
int_cols = ["iterations", "depth", "min_data_in_leaf"]

importance_rows = []
for h, tcol in enumerate(targets, start=1):
    tuned_row = tuned_params_df[tuned_params_df["H"] == h].iloc[0]
    param_cols = [c for c in tuned_params_df.columns if c not in ("H", "Target", "inner_cv_rmse")]
    params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
    for k in int_cols:
        if k in params:
            params[k] = int(params[k])

    if "fitted_models" in dir() and h in fitted_models:
        m = fitted_models[h]
    else:
        y_train_delta = Y_train_abs[tcol].to_numpy() - X_train[QIN_COL].to_numpy()
        y_val_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
        m = CatBoostAdapter.build(params, with_early_stopping=True)
        m, _ = CatBoostAdapter.fit(m, X_train, y_train_delta, X_val, y_val_delta)

    imp = m.get_feature_importance()
    for feat, val in zip(X_train.columns, imp):
        importance_rows.append({"H": h, "hours_ahead": h * 6, "feature": feat, "importance": val})

imp_df = pd.DataFrame(importance_rows)
pivot = imp_df.pivot(index="feature", columns="H", values="importance")
pivot.columns = [f"H{h}({h*6}h)" for h in pivot.columns]
pivot["mean_importance"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean_importance", ascending=False)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("=== Feature importance (PredictionValuesChange, %) ต่อ horizon -- v2 ===")
print(pivot.round(2).to_string())

imp_df.to_csv(os.path.join(OUTDIR, "feature_importance_per_horizon.csv"), index=False)
pivot.to_csv(os.path.join(OUTDIR, "feature_importance_summary.csv"))
print(f"\nบันทึกไว้ที่ {OUTDIR}/feature_importance_per_horizon.csv และ feature_importance_summary.csv")